In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd
import numpy as np
from scipy import sparse

BASE = "/content/drive/MyDrive/NewsGuard"

# Day 2 — Preprocessed data
train_df = pd.read_csv(f"{BASE}/data/processed/train_preprocessed.csv")
val_df = pd.read_csv(f"{BASE}/data/processed/validation_preprocessed.csv")
test_df = pd.read_csv(f"{BASE}/data/processed/test_preprocessed.csv")

# Day 3 — TF-IDF
X_train_tfidf = sparse.load_npz(f"{BASE}/features/tfidf/X_train_tfidf.npz")
X_val_tfidf = sparse.load_npz(f"{BASE}/features/tfidf/X_validation_tfidf.npz")
X_test_tfidf = sparse.load_npz(f"{BASE}/features/tfidf/X_test_tfidf.npz")

# Day 4 — Auxiliary features
X_train_aux = np.load(f"{BASE}/features/auxiliary/X_train_auxiliary.npy")
X_val_aux = np.load(f"{BASE}/features/auxiliary/X_validation_auxiliary.npy")
X_test_aux = np.load(f"{BASE}/features/auxiliary/X_test_auxiliary.npy")

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

print("===== DAY 5 ARTIFACT RESTORE =====")

print("\nData:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTF-IDF:")
print("Train:", X_train_tfidf.shape)
print("Validation:", X_val_tfidf.shape)
print("Test:", X_test_tfidf.shape)

print("\nAuxiliary:")
print("Train:", X_train_aux.shape)
print("Validation:", X_val_aux.shape)
print("Test:", X_test_aux.shape)

print("\nLabels:")
print("Train:", y_train.shape)
print("Validation:", y_val.shape)
print("Test:", y_test.shape)

assert X_train_tfidf.shape == (31282, 5000)
assert X_val_tfidf.shape == (3910, 5000)
assert X_test_tfidf.shape == (3911, 5000)

assert X_train_aux.shape == (31282, 115)
assert X_val_aux.shape == (3910, 115)
assert X_test_aux.shape == (3911, 115)

assert y_train.shape[0] == 31282
assert y_val.shape[0] == 3910
assert y_test.shape[0] == 3911

assert np.isfinite(X_train_aux).all()
assert np.isfinite(X_val_aux).all()
assert np.isfinite(X_test_aux).all()

print("\n✅ DAY 1–4 ARTIFACT RESTORE & VERIFICATION PASSED")

===== DAY 5 ARTIFACT RESTORE =====

Data:
Train: (31282, 7)
Validation: (3910, 7)
Test: (3911, 7)

TF-IDF:
Train: (31282, 5000)
Validation: (3910, 5000)
Test: (3911, 5000)

Auxiliary:
Train: (31282, 115)
Validation: (3910, 115)
Test: (3911, 115)

Labels:
Train: (31282,)
Validation: (3910,)
Test: (3911,)

✅ DAY 1–4 ARTIFACT RESTORE & VERIFICATION PASSED


In [3]:
from scipy.sparse import csr_matrix, hstack

# Convert dense auxiliary features to sparse matrices
X_train_aux_sparse = csr_matrix(X_train_aux)
X_val_aux_sparse = csr_matrix(X_val_aux)
X_test_aux_sparse = csr_matrix(X_test_aux)

# Combine TF-IDF + Auxiliary features
X_train_combined = hstack(
    [X_train_tfidf, X_train_aux_sparse],
    format="csr"
)

X_val_combined = hstack(
    [X_val_tfidf, X_val_aux_sparse],
    format="csr"
)

X_test_combined = hstack(
    [X_test_tfidf, X_test_aux_sparse],
    format="csr"
)

print("===== COMBINED FEATURE MATRICES =====")

print("Train:", X_train_combined.shape)
print("Validation:", X_val_combined.shape)
print("Test:", X_test_combined.shape)

print("\nFeature count:")
print("TF-IDF features:", X_train_tfidf.shape[1])
print("Auxiliary features:", X_train_aux.shape[1])
print("Total features:", X_train_combined.shape[1])

assert X_train_combined.shape == (31282, 5115)
assert X_val_combined.shape == (3910, 5115)
assert X_test_combined.shape == (3911, 5115)

assert X_train_combined.format == "csr"
assert X_val_combined.format == "csr"
assert X_test_combined.format == "csr"

print("\n✅ TF-IDF + AUXILIARY FEATURES COMBINED SUCCESSFULLY")

===== COMBINED FEATURE MATRICES =====
Train: (31282, 5115)
Validation: (3910, 5115)
Test: (3911, 5115)

Feature count:
TF-IDF features: 5000
Auxiliary features: 115
Total features: 5115

✅ TF-IDF + AUXILIARY FEATURES COMBINED SUCCESSFULLY


In [4]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix

# Scale only auxiliary features
# TF-IDF is already appropriately scaled by TfidfVectorizer.
aux_scaler = StandardScaler()

X_train_aux_scaled = aux_scaler.fit_transform(X_train_aux)
X_val_aux_scaled = aux_scaler.transform(X_val_aux)
X_test_aux_scaled = aux_scaler.transform(X_test_aux)

# Convert scaled auxiliary features to sparse
X_train_aux_scaled_sparse = csr_matrix(X_train_aux_scaled)
X_val_aux_scaled_sparse = csr_matrix(X_val_aux_scaled)
X_test_aux_scaled_sparse = csr_matrix(X_test_aux_scaled)

# Rebuild combined matrices
X_train_combined = hstack(
    [X_train_tfidf, X_train_aux_scaled_sparse],
    format="csr"
)

X_val_combined = hstack(
    [X_val_tfidf, X_val_aux_scaled_sparse],
    format="csr"
)

X_test_combined = hstack(
    [X_test_tfidf, X_test_aux_scaled_sparse],
    format="csr"
)

print("===== SCALED COMBINED FEATURES =====")
print("Train:", X_train_combined.shape)
print("Validation:", X_val_combined.shape)
print("Test:", X_test_combined.shape)

print("\nAuxiliary scaled mean:",
      np.mean(X_train_aux_scaled, axis=0).mean())

print("Auxiliary scaled std:",
      np.std(X_train_aux_scaled, axis=0).mean())

assert X_train_combined.shape == (31282, 5115)
assert X_val_combined.shape == (3910, 5115)
assert X_test_combined.shape == (3911, 5115)

assert np.isfinite(X_train_aux_scaled).all()
assert np.isfinite(X_val_aux_scaled).all()
assert np.isfinite(X_test_aux_scaled).all()

print("\n✅ AUXILIARY FEATURES SCALED WITHOUT DATA LEAKAGE")

===== SCALED COMBINED FEATURES =====
Train: (31282, 5115)
Validation: (3910, 5115)
Test: (3911, 5115)

Auxiliary scaled mean: 2.5114246e-09
Auxiliary scaled std: 0.9999986

✅ AUXILIARY FEATURES SCALED WITHOUT DATA LEAKAGE


In [5]:
import joblib

combined_dir = f"{BASE}/features/combined"
os.makedirs(combined_dir, exist_ok=True)

# Save combined feature matrices
sparse.save_npz(
    f"{combined_dir}/X_train_combined.npz",
    X_train_combined
)

sparse.save_npz(
    f"{combined_dir}/X_validation_combined.npz",
    X_val_combined
)

sparse.save_npz(
    f"{combined_dir}/X_test_combined.npz",
    X_test_combined
)

# Save auxiliary scaler
joblib.dump(
    aux_scaler,
    f"{combined_dir}/auxiliary_scaler.joblib"
)

print("===== SAVED COMBINED FEATURES =====")

print(os.path.getsize(
    f"{combined_dir}/X_train_combined.npz"
) / (1024**2), "MB - Train")

print(os.path.getsize(
    f"{combined_dir}/X_validation_combined.npz"
) / (1024**2), "MB - Validation")

print(os.path.getsize(
    f"{combined_dir}/X_test_combined.npz"
) / (1024**2), "MB - Test")

print("\nScaler saved:")
print(f"{combined_dir}/auxiliary_scaler.joblib")

print("\n✅ COMBINED FEATURE ARTIFACTS SAVED")

===== SAVED COMBINED FEATURES =====
50.270344734191895 MB - Train
6.118110656738281 MB - Validation
6.0973663330078125 MB - Test

Scaler saved:
/content/drive/MyDrive/NewsGuard/features/combined/auxiliary_scaler.joblib

✅ COMBINED FEATURE ARTIFACTS SAVED


In [6]:
from sklearn.pipeline import FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from scipy import sparse
import joblib
import numpy as np


class PrecomputedFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, matrix):
        self.matrix = matrix

    def fit(self, X=None, y=None):
        return self

    def transform(self, X=None):
        return self.matrix


feature_union = FeatureUnion([
    (
        "tfidf",
        PrecomputedFeatureTransformer(X_train_tfidf)
    ),
    (
        "auxiliary",
        PrecomputedFeatureTransformer(X_train_aux_scaled_sparse)
    )
])

X_train_union = feature_union.fit_transform(None)

print("===== FEATUREUNION PIPELINE =====")
print("TF-IDF features:", X_train_tfidf.shape[1])
print("Auxiliary features:", X_train_aux_scaled_sparse.shape[1])
print("Combined features:", X_train_union.shape)
print("Format:", X_train_union.format)

assert X_train_union.shape == (31282, 5115)
assert sparse.issparse(X_train_union)

print("\n✅ FEATUREUNION PIPELINE CREATED")

===== FEATUREUNION PIPELINE =====
TF-IDF features: 5000
Auxiliary features: 115
Combined features: (31282, 5115)
Format: csr

✅ FEATUREUNION PIPELINE CREATED


In [7]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score, roc_auc_score

# Use the already-created combined sparse training matrix
X_train_cv = X_train_combined
y_train_cv = y_train

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc"
}

print("===== DAY 5 CROSS-VALIDATION SETUP =====")
print("Training matrix:", X_train_cv.shape)
print("CV folds:", cv.n_splits)
print("Shuffle:", cv.shuffle)
print("Random state:", cv.random_state)

print("\nScoring metrics:")
print(list(scoring.keys()))

assert X_train_cv.shape == (31282, 5115)
assert y_train_cv.shape == (31282,)

print("\n✅ STRATIFIED 5-FOLD CV READY")

===== DAY 5 CROSS-VALIDATION SETUP =====
Training matrix: (31282, 5115)
CV folds: 5
Shuffle: True
Random state: 42

Scoring metrics:
['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

✅ STRATIFIED 5-FOLD CV READY


In [8]:
logistic_model = LogisticRegression(
    max_iter=1000,
    solver="liblinear",
    random_state=42
)

print("===== LOGISTIC REGRESSION — 5-FOLD CV =====")

logistic_cv = cross_validate(
    logistic_model,
    X_train_cv,
    y_train_cv,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

print("\nFold-wise Results:")

for fold in range(5):
    print(
        f"Fold {fold + 1}: "
        f"Accuracy={logistic_cv['test_accuracy'][fold]:.4f}, "
        f"Precision={logistic_cv['test_precision'][fold]:.4f}, "
        f"Recall={logistic_cv['test_recall'][fold]:.4f}, "
        f"F1={logistic_cv['test_f1'][fold]:.4f}, "
        f"ROC-AUC={logistic_cv['test_roc_auc'][fold]:.4f}"
    )

print("\n===== LOGISTIC REGRESSION MEAN =====")

print(f"Accuracy : {logistic_cv['test_accuracy'].mean():.4f}")
print(f"Precision: {logistic_cv['test_precision'].mean():.4f}")
print(f"Recall   : {logistic_cv['test_recall'].mean():.4f}")
print(f"F1       : {logistic_cv['test_f1'].mean():.4f}")
print(f"ROC-AUC  : {logistic_cv['test_roc_auc'].mean():.4f}")

print("\n===== LOGISTIC REGRESSION STD =====")

print(f"Accuracy : {logistic_cv['test_accuracy'].std():.4f}")
print(f"Precision: {logistic_cv['test_precision'].std():.4f}")
print(f"Recall   : {logistic_cv['test_recall'].std():.4f}")
print(f"F1       : {logistic_cv['test_f1'].std():.4f}")
print(f"ROC-AUC  : {logistic_cv['test_roc_auc'].std():.4f}")

print("\n✅ LOGISTIC REGRESSION 5-FOLD CV COMPLETED")

===== LOGISTIC REGRESSION — 5-FOLD CV =====

Fold-wise Results:
Fold 1: Accuracy=0.9936, Precision=0.9947, Recall=0.9935, F1=0.9941, ROC-AUC=0.9997
Fold 2: Accuracy=0.9934, Precision=0.9944, Recall=0.9935, F1=0.9940, ROC-AUC=0.9995
Fold 3: Accuracy=0.9931, Precision=0.9935, Recall=0.9938, F1=0.9937, ROC-AUC=0.9994
Fold 4: Accuracy=0.9930, Precision=0.9929, Recall=0.9941, F1=0.9935, ROC-AUC=0.9993
Fold 5: Accuracy=0.9938, Precision=0.9921, Recall=0.9965, F1=0.9943, ROC-AUC=0.9991

===== LOGISTIC REGRESSION MEAN =====
Accuracy : 0.9934
Precision: 0.9935
Recall   : 0.9943
F1       : 0.9939
ROC-AUC  : 0.9994

===== LOGISTIC REGRESSION STD =====
Accuracy : 0.0003
Precision: 0.0010
Recall   : 0.0011
F1       : 0.0003
ROC-AUC  : 0.0002

✅ LOGISTIC REGRESSION 5-FOLD CV COMPLETED


In [13]:
import numpy as np
import os

AUX_DIR = "/content/drive/MyDrive/NewsGuard/features/auxiliary"

X_train_aux = np.load(
    os.path.join(AUX_DIR, "X_train_auxiliary.npy")
)

X_validation_aux = np.load(
    os.path.join(AUX_DIR, "X_validation_auxiliary.npy")
)

X_test_aux = np.load(
    os.path.join(AUX_DIR, "X_test_auxiliary.npy")
)

print("===== DAY 4 AUXILIARY FEATURES RESTORED =====")
print("Train:", X_train_aux.shape)
print("Validation:", X_validation_aux.shape)
print("Test:", X_test_aux.shape)

print("\nData type:", X_train_aux.dtype)

assert X_train_aux.shape == (31282, 115)
assert X_validation_aux.shape == (3910, 115)
assert X_test_aux.shape == (3911, 115)

assert np.isfinite(X_train_aux).all()
assert np.isfinite(X_validation_aux).all()
assert np.isfinite(X_test_aux).all()

print("\n✅ DAY 4 AUXILIARY FEATURES RESTORED & VERIFIED")

===== DAY 4 AUXILIARY FEATURES RESTORED =====
Train: (31282, 115)
Validation: (3910, 115)
Test: (3911, 115)

Data type: float32

✅ DAY 4 AUXILIARY FEATURES RESTORED & VERIFIED


In [15]:
from scipy import sparse
import os

TFIDF_DIR = "/content/drive/MyDrive/NewsGuard/features/tfidf"

X_train_tfidf = sparse.load_npz(
    os.path.join(TFIDF_DIR, "X_train_tfidf.npz")
)

X_validation_tfidf = sparse.load_npz(
    os.path.join(TFIDF_DIR, "X_validation_tfidf.npz")
)

X_test_tfidf = sparse.load_npz(
    os.path.join(TFIDF_DIR, "X_test_tfidf.npz")
)

print("===== DAY 3 TF-IDF FEATURES RESTORED =====")
print("Train:", X_train_tfidf.shape)
print("Validation:", X_validation_tfidf.shape)
print("Test:", X_test_tfidf.shape)

assert X_train_tfidf.shape == (31282, 5000)
assert X_validation_tfidf.shape == (3910, 5000)
assert X_test_tfidf.shape == (3911, 5000)

assert sparse.issparse(X_train_tfidf)
assert sparse.issparse(X_validation_tfidf)
assert sparse.issparse(X_test_tfidf)

print("\n✅ DAY 3 TF-IDF FEATURES RESTORED & VERIFIED")

===== DAY 3 TF-IDF FEATURES RESTORED =====
Train: (31282, 5000)
Validation: (3910, 5000)
Test: (3911, 5000)

✅ DAY 3 TF-IDF FEATURES RESTORED & VERIFIED


In [17]:
import numpy as np

train_negative = X_train_nb.data[X_train_nb.data < 0]
validation_negative = X_validation_nb.data[X_validation_nb.data < 0]
test_negative = X_test_nb.data[X_test_nb.data < 0]

print("===== NEGATIVE VALUE CHECK =====")

print("Train negative values:", len(train_negative))
print("Validation negative values:", len(validation_negative))
print("Test negative values:", len(test_negative))

if len(validation_negative) > 0:
    print("\nValidation minimum:", validation_negative.min())

if len(test_negative) > 0:
    print("Test minimum:", test_negative.min())

print("\n✅ NEGATIVE VALUE MAGNITUDE CHECK COMPLETED")

===== NEGATIVE VALUE CHECK =====
Train negative values: 0
Validation negative values: 6
Test negative values: 10

Validation minimum: -0.09883098
Test minimum: -0.26548344

✅ NEGATIVE VALUE MAGNITUDE CHECK COMPLETED


In [19]:
from sklearn.preprocessing import MinMaxScaler
from scipy import sparse
import numpy as np
import joblib

# Fit scaler ONLY on training auxiliary features
nb_aux_scaler = MinMaxScaler()

X_train_aux_nb = nb_aux_scaler.fit_transform(X_train_aux)

X_validation_aux_nb = nb_aux_scaler.transform(X_validation_aux)
X_test_aux_nb = nb_aux_scaler.transform(X_test_aux)

# Clip values outside the training range
X_train_aux_nb = np.clip(X_train_aux_nb, 0, 1)
X_validation_aux_nb = np.clip(X_validation_aux_nb, 0, 1)
X_test_aux_nb = np.clip(X_test_aux_nb, 0, 1)

# Convert to sparse
X_train_aux_nb_sparse = sparse.csr_matrix(
    X_train_aux_nb.astype(np.float32)
)

X_validation_aux_nb_sparse = sparse.csr_matrix(
    X_validation_aux_nb.astype(np.float32)
)

X_test_aux_nb_sparse = sparse.csr_matrix(
    X_test_aux_nb.astype(np.float32)
)

# Combine TF-IDF + auxiliary features
X_train_nb = sparse.hstack(
    [X_train_tfidf, X_train_aux_nb_sparse],
    format="csr"
)

X_validation_nb = sparse.hstack(
    [X_validation_tfidf, X_validation_aux_nb_sparse],
    format="csr"
)

X_test_nb = sparse.hstack(
    [X_test_tfidf, X_test_aux_nb_sparse],
    format="csr"
)

print("===== FINAL NAIVE BAYES FEATURE SET =====")
print("Train:", X_train_nb.shape)
print("Validation:", X_validation_nb.shape)
print("Test:", X_test_nb.shape)

print("\nMinimum values:")
print("Train:", X_train_nb.min())
print("Validation:", X_validation_nb.min())
print("Test:", X_test_nb.min())

print("\nMaximum values:")
print("Train:", X_train_nb.max())
print("Validation:", X_validation_nb.max())
print("Test:", X_test_nb.max())

assert X_train_nb.shape == (31282, 5115)
assert X_validation_nb.shape == (3910, 5115)
assert X_test_nb.shape == (3911, 5115)

assert X_train_nb.min() >= 0
assert X_validation_nb.min() >= 0
assert X_test_nb.min() >= 0

joblib.dump(
    nb_aux_scaler,
    "/content/drive/MyDrive/NewsGuard/features/combined/nb_auxiliary_minmax_scaler.joblib"
)

print("\n✅ FINAL NAIVE BAYES FEATURES VERIFIED")

===== FINAL NAIVE BAYES FEATURE SET =====
Train: (31282, 5115)
Validation: (3910, 5115)
Test: (3911, 5115)

Minimum values:
Train: 0.0
Validation: 0.0
Test: 0.0

Maximum values:
Train: 1.0
Validation: 1.0
Test: 1.0

✅ FINAL NAIVE BAYES FEATURES VERIFIED


In [20]:
from sklearn.naive_bayes import MultinomialNB

naive_bayes_model = MultinomialNB()

print("===== NAIVE BAYES — 5-FOLD CV =====")

naive_bayes_cv = cross_validate(
    naive_bayes_model,
    X_train_nb,
    y_train_cv,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

print("\nFold-wise Results:")

for fold in range(5):
    print(
        f"Fold {fold + 1}: "
        f"Accuracy={naive_bayes_cv['test_accuracy'][fold]:.4f}, "
        f"Precision={naive_bayes_cv['test_precision'][fold]:.4f}, "
        f"Recall={naive_bayes_cv['test_recall'][fold]:.4f}, "
        f"F1={naive_bayes_cv['test_f1'][fold]:.4f}, "
        f"ROC-AUC={naive_bayes_cv['test_roc_auc'][fold]:.4f}"
    )

print("\n===== NAIVE BAYES MEAN =====")

print(f"Accuracy : {naive_bayes_cv['test_accuracy'].mean():.4f}")
print(f"Precision: {naive_bayes_cv['test_precision'].mean():.4f}")
print(f"Recall   : {naive_bayes_cv['test_recall'].mean():.4f}")
print(f"F1       : {naive_bayes_cv['test_f1'].mean():.4f}")
print(f"ROC-AUC  : {naive_bayes_cv['test_roc_auc'].mean():.4f}")

print("\n===== NAIVE BAYES STD =====")

print(f"Accuracy : {naive_bayes_cv['test_accuracy'].std():.4f}")
print(f"Precision: {naive_bayes_cv['test_precision'].std():.4f}")
print(f"Recall   : {naive_bayes_cv['test_recall'].std():.4f}")
print(f"F1       : {naive_bayes_cv['test_f1'].std():.4f}")
print(f"ROC-AUC  : {naive_bayes_cv['test_roc_auc'].std():.4f}")

print("\n✅ NAIVE BAYES 5-FOLD CV COMPLETED")

===== NAIVE BAYES — 5-FOLD CV =====

Fold-wise Results:
Fold 1: Accuracy=0.9527, Precision=0.9542, Recall=0.9587, F1=0.9565, ROC-AUC=0.9898
Fold 2: Accuracy=0.9537, Precision=0.9564, Recall=0.9581, F1=0.9573, ROC-AUC=0.9910
Fold 3: Accuracy=0.9527, Precision=0.9497, Recall=0.9637, F1=0.9567, ROC-AUC=0.9898
Fold 4: Accuracy=0.9514, Precision=0.9509, Recall=0.9599, F1=0.9554, ROC-AUC=0.9893
Fold 5: Accuracy=0.9560, Precision=0.9526, Recall=0.9670, F1=0.9598, ROC-AUC=0.9913

===== NAIVE BAYES MEAN =====
Accuracy : 0.9533
Precision: 0.9528
Recall   : 0.9615
F1       : 0.9571
ROC-AUC  : 0.9902

===== NAIVE BAYES STD =====
Accuracy : 0.0015
Precision: 0.0024
Recall   : 0.0034
F1       : 0.0015
ROC-AUC  : 0.0008

✅ NAIVE BAYES 5-FOLD CV COMPLETED


In [21]:
import pandas as pd
import joblib
import os

baseline_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Naive Bayes"
    ],
    "CV_Accuracy": [
        logistic_cv["test_accuracy"].mean(),
        naive_bayes_cv["test_accuracy"].mean()
    ],
    "CV_Precision": [
        logistic_cv["test_precision"].mean(),
        naive_bayes_cv["test_precision"].mean()
    ],
    "CV_Recall": [
        logistic_cv["test_recall"].mean(),
        naive_bayes_cv["test_recall"].mean()
    ],
    "CV_F1": [
        logistic_cv["test_f1"].mean(),
        naive_bayes_cv["test_f1"].mean()
    ],
    "CV_ROC_AUC": [
        logistic_cv["test_roc_auc"].mean(),
        naive_bayes_cv["test_roc_auc"].mean()
    ]
})

baseline_results = baseline_results.sort_values(
    by="CV_F1",
    ascending=False
).reset_index(drop=True)

RESULTS_DIR = "/content/drive/MyDrive/NewsGuard/results"

os.makedirs(RESULTS_DIR, exist_ok=True)

baseline_results.to_csv(
    os.path.join(RESULTS_DIR, "day_05_baseline_results.csv"),
    index=False
)

joblib.dump(
    baseline_results,
    os.path.join(RESULTS_DIR, "day_05_baseline_results.joblib")
)

print("===== DAY 5 BASELINE MODEL COMPARISON =====")
print(baseline_results.to_string(index=False))

print("\n🏆 CURRENT BEST BASELINE:")
print(baseline_results.iloc[0]["Model"])

print(
    f"Best CV F1: "
    f"{baseline_results.iloc[0]['CV_F1']:.4f}"
)

print("\n✅ DAY 5 BASELINE RESULTS SAVED")

===== DAY 5 BASELINE MODEL COMPARISON =====
              Model  CV_Accuracy  CV_Precision  CV_Recall    CV_F1  CV_ROC_AUC
Logistic Regression     0.993383      0.993520   0.994280 0.993899    0.999408
        Naive Bayes     0.953296      0.952792   0.961491 0.957115    0.990242

🏆 CURRENT BEST BASELINE:
Logistic Regression
Best CV F1: 0.9939

✅ DAY 5 BASELINE RESULTS SAVED
